# Cursor Remote SSH + CUDA workbook

This is the complete remote workflow: connect to a Linux/NVIDIA host, prepare a remote checkout, start a remote folder-local kernel, verify CUDA, run PyTorch, and release the GPU.

## Before opening Cursor

On the Mac, add a real host to `~/.ssh/config`:

```sshconfig
Host gpu-box
    HostName YOUR_GPU_HOST
    User YOUR_REMOTE_USER
    IdentityFile ~/.ssh/id_ed25519
```

Test SSH and the GPU directly with `ssh gpu-box` and `nvidia-smi`. On the remote host, from the remote checkout, run `uv sync`. If this is a new checkout, clone it first.

## Start the remote kernel in Cursor

1. Run `Remote-SSH: Connect to Host...` and choose `gpu-box`.
2. Open the remote `cursor-notebook-workspace` folder.
3. Open this notebook from the remote Explorer.
4. In the notebook toolbar, click `Select Kernel` (or the current kernel name).
5. Choose `Select Another Kernel...`.
6. Choose `Python Environments...`, then choose the remote `.venv/bin/python`.
7. Click `Run All`. The first code cell starts the kernel process on the remote machine.

## Keyboard controls on macOS

- Execute the selected cell: `Control+Enter` (`⌃⏎`).
- Add a code cell: `Command+Enter` (`⌘⏎`).
- Execute everything: click `Run All` in the notebook toolbar.

![Cursor notebook kernel picker flow](../docs/images/cursor-kernel-picker-flow.svg)

Do not use the Command Palette's `Jupyter: Manage Access To Jupyter Kernels`. There is no file-sync layer: this notebook and kernel are both remote.

## What success looks like

The first cell prints a remote `.venv` executable, a CUDA version, and an NVIDIA GPU name. The next cells test CUDA matrix multiplication and autograd. This notebook intentionally fails if CUDA is unavailable; it never falls back to CPU or Mac MPS.

## Stop and release resources

Commit or push code, close the notebook, disconnect Remote SSH, and stop or terminate the GPU instance with its provider. Closing Cursor does not stop a rented GPU.

In [ ]:
import platform
import sys

import torch

# This cell runs in the remote Linux kernel selected through Cursor Remote SSH.
# Fail immediately instead of silently running the CUDA notebook on CPU/MPS.
assert torch.cuda.is_available(), "This notebook requires an NVIDIA CUDA runtime."
device = torch.device("cuda")
print(
    {
        "platform": platform.platform(),
        "executable": sys.executable,
        "torch": torch.__version__,
        "cuda": torch.version.cuda,
        "device": torch.cuda.get_device_name(0),
    }
)

In [ ]:
left = torch.ones((1024, 1024), device=device)
right = torch.full((1024, 1024), 2.0, device=device)
product = left @ right
torch.cuda.synchronize()
print(
    {
        "device": str(product.device),
        "shape": tuple(product.shape),
        "sample": float(product[0, 0].cpu()),
    }
)

In [ ]:
weights = torch.randn(1024, 1024, device=device, requires_grad=True)
loss = weights.square().mean()
loss.backward()
torch.cuda.synchronize()
print(
    {"loss": loss.detach().cpu().item(), "gradient_norm": weights.grad.norm().detach().cpu().item()}
)